In [1]:
# ============================================================
# Notebook : 07_crossval_generalization.ipynb
# Project  : Causal-Guided Multimodal Sensor Fusion for Elderly Risk Detection
# Purpose  : Aggregate the household-level Group 5-fold results, report
#            mean +/- standard deviation for every model, run paired
#            significance tests of the proposed model against each baseline,
#            and characterise cross-household generalization via per-household
#            AUC. Produces the main model-comparison figure (grayscale, with
#            error bars) and a per-household performance figure.
# Note     : Anonymous repository for double-blind review.
# ============================================================

from pathlib import Path
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.metrics import roc_auc_score

ROOT     = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TAB_DIR  = ROOT / "results" / "tables"; FIG_DIR = ROOT / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper")
mpl.rcParams.update({"figure.dpi":150,"savefig.dpi":600,"savefig.bbox":"tight",
                     "font.family":"DejaVu Sans","axes.unicode_minus":False})
def savefig(fig, name):
    fig.tight_layout()
    for ext in ("png","pdf"): fig.savefig(FIG_DIR/f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)
GC = {"d":"#222222","m":"#777777","l":"#BBBBBB"}

# ── Combine proposed + baselines ─────────────────────────────
prop = pd.read_csv(TAB_DIR / "cv_proposed.csv").rename(columns={"model":"model"})
prop["model"] = "Proposed (Causal+Lag)"
base = pd.read_csv(TAB_DIR / "cv_baselines.csv")
cv = pd.concat([base, prop[["model","fold","macro_f1","binary_f1","auc_roc"]]], ignore_index=True)

ORDER = ["Single-Modal MLP (EM)","Early Fusion MLP","Attention Fusion",
         "Transformer Fusion","Proposed (Causal+Lag)"]
cv["model"] = pd.Categorical(cv["model"], categories=ORDER, ordered=True)

# ── Summary table (mean +/- std) ─────────────────────────────
summ = (cv.groupby("model", observed=True)[["macro_f1","binary_f1","auc_roc"]]
          .agg(["mean","std"]).round(4))
summ.columns = ["_".join(c) for c in summ.columns]
summ = summ.reset_index()
summ.to_csv(TAB_DIR / "cv_summary.csv", index=False)
print("Cross-validation summary (mean +/- std over 5 folds):")
for _, r in summ.iterrows():
    print(f"  {r['model']:<24} AUC {r.auc_roc_mean:.3f}+/-{r.auc_roc_std:.3f} | "
          f"macroF1 {r.macro_f1_mean:.3f}+/-{r.macro_f1_std:.3f} | "
          f"binF1 {r.binary_f1_mean:.3f}+/-{r.binary_f1_std:.3f}")

# ── Paired significance tests: proposed vs each baseline ─────
prop_auc = cv[cv.model == "Proposed (Causal+Lag)"].sort_values("fold")["auc_roc"].values
sig_rows = []
for b in ORDER[:-1]:
    b_auc = cv[cv.model == b].sort_values("fold")["auc_roc"].values
    t_stat, t_p = stats.ttest_rel(prop_auc, b_auc)
    try:
        w_stat, w_p = stats.wilcoxon(prop_auc, b_auc)
    except Exception:
        w_p = np.nan
    sig_rows.append({"comparison": f"Proposed vs {b}",
                     "mean_auc_diff": round(float(prop_auc.mean() - b_auc.mean()), 4),
                     "paired_t_p": round(float(t_p), 4),
                     "wilcoxon_p": (round(float(w_p), 4) if w_p == w_p else np.nan)})
sig = pd.DataFrame(sig_rows); sig.to_csv(TAB_DIR / "significance_tests.csv", index=False)
print("\nPaired tests (proposed vs baseline, AUC over folds):")
for _, r in sig.iterrows():
    print(f"  {r.comparison:<34} dAUC={r.mean_auc_diff:+.3f}  t-p={r.paired_t_p}  wilcoxon-p={r.wilcoxon_p}")

# ── Fig: model comparison (AUC / macroF1 / binF1, error bars) ─
metrics = [("auc_roc","AUC-ROC"),("macro_f1","Macro F1"),("binary_f1","Binary F1")]
fig, ax = plt.subplots(figsize=(8.5, 4))
xpos = np.arange(len(ORDER)); w = 0.26
grays = [GC["l"], GC["m"], GC["d"]]
for k,(mcol,mlab) in enumerate(metrics):
    means = [cv[cv.model==m][mcol].mean() for m in ORDER]
    stds  = [cv[cv.model==m][mcol].std()  for m in ORDER]
    ax.bar(xpos + (k-1)*w, means, w, yerr=stds, capsize=3, label=mlab,
           color=grays[k], edgecolor="black", linewidth=0.6)
ax.axhline(0.5, color="black", ls="--", lw=0.8)
ax.set_xticks(xpos); ax.set_xticklabels([m.replace(" ","\n") for m in ORDER], fontsize=8)
ax.set_ylabel("Score"); ax.set_ylim(0, 1.0); ax.legend(fontsize=8, ncol=3, loc="upper left")
savefig(fig, "fig07_model_comparison_cv")

# ── Per-household AUC (proposed) ─────────────────────────────
pred = pd.read_csv(TAB_DIR / "pred_proposed.csv")
hh_rows = []
for hh, g in pred.groupby("HH_ID"):
    if g.y_true.nunique() < 2:
        auc = np.nan
    else:
        auc = roc_auc_score(g.y_true, g.y_prob)
    hh_rows.append({"HH_ID": hh, "fold": int(g.fold.iloc[0]),
                    "n": len(g), "risk_rate": round(g.y_true.mean(),3),
                    "auc": (round(float(auc),3) if auc==auc else np.nan)})
hh_df = pd.DataFrame(hh_rows).sort_values("auc")
hh_df.to_csv(TAB_DIR / "per_household_auc.csv", index=False)
valid = hh_df.auc.dropna()
print(f"\nPer-household AUC (proposed): mean {valid.mean():.3f} +/- {valid.std():.3f} "
      f"(min {valid.min():.3f}, max {valid.max():.3f}, n={len(valid)})")

fig, ax = plt.subplots(figsize=(7, 3.4))
xs = np.arange(len(valid))
ax.bar(xs, sorted(valid.values), color=GC["m"], edgecolor="black", linewidth=0.5)
ax.axhline(0.5, color="black", ls="--", lw=0.8)
ax.axhline(valid.mean(), color=GC["d"], ls=":", lw=1.2)
ax.set_xlabel("Household (sorted)"); ax.set_ylabel("AUC-ROC"); ax.set_ylim(0, 1.0)
savefig(fig, "fig08_per_household_auc")

print("Saved summary, significance, per-household tables and figures.")
print("07_crossval_generalization complete.")


Cross-validation summary (mean +/- std over 5 folds):
  Single-Modal MLP (EM)    AUC 0.678+/-0.110 | macroF1 0.578+/-0.089 | binF1 0.453+/-0.086
  Early Fusion MLP         AUC 0.738+/-0.149 | macroF1 0.664+/-0.069 | binF1 0.488+/-0.132
  Attention Fusion         AUC 0.751+/-0.132 | macroF1 0.675+/-0.066 | binF1 0.505+/-0.126
  Transformer Fusion       AUC 0.754+/-0.147 | macroF1 0.671+/-0.074 | binF1 0.501+/-0.145
  Proposed (Causal+Lag)    AUC 0.709+/-0.123 | macroF1 0.622+/-0.066 | binF1 0.422+/-0.139

Paired tests (proposed vs baseline, AUC over folds):
  Proposed vs Single-Modal MLP (EM)  dAUC=+0.032  t-p=0.2396  wilcoxon-p=0.3125
  Proposed vs Early Fusion MLP       dAUC=-0.028  t-p=0.2347  wilcoxon-p=0.4375
  Proposed vs Attention Fusion       dAUC=-0.042  t-p=0.0932  wilcoxon-p=0.125
  Proposed vs Transformer Fusion     dAUC=-0.044  t-p=0.2229  wilcoxon-p=0.3125



Per-household AUC (proposed): mean 0.722 +/- 0.121 (min 0.455, max 0.900, n=25)


Saved summary, significance, per-household tables and figures.
07_crossval_generalization complete.
